In [1]:
# Run once: install pyyaml if not already installed
!pip install pyyaml

In [2]:

#Finding where am I
import os
os.environ['USE_LOCALSTACK'] = '1'
os.environ['LOCALSTACK_URL'] = 'http://192.168.1.30:4566'
os.chdir('/config/workspace/python/pythonawsproject')
print(os.getcwd())

# Now import (after chdir so Python can find the network package)
from network.state import load_state
print(load_state())

/config/workspace/python/pythonawsproject
{}


In [4]:
#What's the plan yo!
import yaml

with open('config/network.yaml', 'r') as f:
    config = yaml.safe_load(f) #//putting all the values from load state to config

print(config['vpc'])
print(config['security_groups'])

{'name': 'dev-vpc', 'cidr': '10.0.0.0/16', 'region': 'ap-south-1'}
[{'name': 'dev-web-sg', 'description': 'Allow SSH, HTTP, HTTPS inbound', 'ingress': [{'protocol': 'tcp', 'port': 22, 'cidr': '0.0.0.0/0', 'description': 'SSH access'}, {'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}, {'protocol': 'tcp', 'port': 443, 'cidr': '0.0.0.0/0', 'description': 'HTTPs'}], 'egress': [{'protocol': 'tcp', 'port': 80, 'cidr': '0.0.0.0/0', 'description': 'HTTP'}]}]


In [5]:
##Now it can work
from network.vpc import create_vpc

vpc_id = create_vpc(config)
print(f"Result: {vpc_id}")

Created VPC: vpc-086f174b
VPC is now avilable
Tagged VPC with: ['Name', 'ManagedBy', 'Layer', 'Environment']
State saved
Result: vpc-086f174b


In [6]:
from network.subnets import create_subnet
##We are utting full config python file here for execution
result = create_subnet(config)
print(result)

[{'name': 'public-subnet-az1', 'cidr': '10.0.1.0/24', 'az': 'ap-south-1a'}, {'name': 'public-subnet-az2', 'cidr': '10.0.2.0/24', 'az': 'ap-south-1b'}, {'name': 'private-subnet-az1', 'cidr': '10.0.3.0/24', 'az': 'ap-south-1a'}, {'name': 'private-subnet-az2', 'cidr': '10.0.4.0/24', 'az': 'ap-south-1b'}]
{'Subnet': {'AvailabilityZoneId': 'aps1-az1', 'MapCustomerOwnedIpOnLaunch': False, 'OwnerId': '000000000000', 'AssignIpv6AddressOnCreation': False, 'Ipv6CidrBlockAssociationSet': [], 'Tags': [], 'SubnetArn': 'arn:aws:ec2:ap-south-1:000000000000:subnet/subnet-bb5cad5d', 'EnableDns64': False, 'SubnetId': 'subnet-bb5cad5d', 'State': 'available', 'VpcId': 'vpc-086f174b', 'CidrBlock': '10.0.1.0/24', 'AvailableIpAddressCount': 251, 'AvailabilityZone': 'ap-south-1a', 'DefaultForAz': False, 'MapPublicIpOnLaunch': False}, 'ResponseMetadata': {'RequestId': 'a63c3b28-0a82-4521-b745-468e9966d49a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'content-length': '862', 'content-type': 'application/xml', 'date

In [7]:
from network.internet_gateway  import creating_igw
from network.state import load_state, save_state
from network.aws_client import get_client

igw_id = creating_igw(config)
print(f"Result: {igw_id}")
# ec2 = get_client('ec2')
# response = ec2.describe_internet_gateways(
#     Filters=[{'Name': 'attachment.vpc-id', 'Values': [vpc_id]}]
# )

# print(vpc_id)

# response = ec2.describe_internet_gateways()
# print(response['InternetGateways'])

# igw_id = response['InternetGateways'][0]['InternetGatewayId']
# print(f"Found exsisting IGW: {igw_id}")

# ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
# print(f"Attached {igw_id} to {vpc_id}")

# ##Saving and loading of state file
# state = load_state()
# state['igw_id'] = igw_id
# save_state(state)
# print("Saved State")

# igw_id = creating_igw(config)
# print(f"Result: {igw_id}")


Created IGW: igw-ac703f85
Attachment IGW to VPC: vpc-086f174b
Tagged  IGW: dev-igw
Saved IGW to State
Result: igw-ac703f85


In [9]:
from network.route_tables import create_route_tables

result = create_route_tables(config)

Public route table already exsists: rtb-f4da44dc
 Already associated: public-subnet-az1
 Already associated: public-subnet-az2
Private route table already exists: rtb-9425dc59
  Already associated: private-subnet-az1
  Already associated: private-subnet-az2
Route tables done.


In [11]:
from network.nat_gateway import create_nat_gateway

result = create_nat_gateway(config)

Elastic IP already exists: eipalloc-f15f5bb0c19a51730
NAT Gateway already exists: nat-5418ea87da75c0603
Private route to NAT already exists
NAT Gateway done.


In [12]:
from network.state import load_state, save_state

state = load_state()
del state['nat_id']
del state['nat_eip_alloc_id']

In [14]:
import os
os.chdir('/config/workspace/python/pythonawsproject')
print(os.listdir('network/'))


['internet_gateway.py', 'state.py', 'security_groups.py', 'planner.py', 'route_tables.py', 'main.py', 'nat_gateway.py', '__init__.py', 'ec2_instance.py', 'subnets.py', 'aws_client.py', '__pycache__', 'vpc.py']


In [15]:
from network.security_groups import create_security_groups
create_security_groups(config)

Created Security Group: sg-5c85da35302d46462 (dev-web-sg)
  Added 3 ingress rules
  Revoked default egress (allow-all) rule
  Tagged + saved: sg_dev-web-sg = sg-5c85da35302d46462
Security Groups done.


{'vpc_id': 'vpc-086f174b',
 'vpc_name': 'dev-vpc',
 'public-subnet-az1': 'subnet-bb5cad5d',
 'public-subnet-az2': 'subnet-92febc26',
 'private-subnet-az1': 'subnet-d3dcfd50',
 'private-subnet-az2': 'subnet-5cb4937f',
 'igw_id': 'igw-ac703f85',
 'public_rt_id': 'rtb-f4da44dc',
 'public-subnet-az1_rt_assoc': 'rtbassoc-9a37ac81',
 'public-subnet-az2_rt_assoc': 'rtbassoc-b8b898c5',
 'private_rt_id': 'rtb-9425dc59',
 'private-subnet-az1_rt_assoc': 'rtbassoc-c996fd19',
 'private-subnet-az2_rt_assoc': 'rtbassoc-334a6637',
 'nat_eip_alloc_id': 'eipalloc-f15f5bb0c19a51730',
 'nat_id': 'nat-5418ea87da75c0603',
 'private_nat_route': True,
 'sg_dev-web-sg': 'sg-5c85da35302d46462'}

In [21]:
from network.planner import run_planner
plan = run_planner()

🔍 Verifying resources against AWS...

  ✅ No drift — state matches AWS.

  EXECUTION PLAN
  ✅ vpc                  "dev-vpc" → EXISTS (vpc-086f174b...)
  ✅ subnet               "public-subnet-az1" → EXISTS (subnet-bb5cad5d...)
  ✅ subnet               "public-subnet-az2" → EXISTS (subnet-92febc26...)
  ✅ subnet               "private-subnet-az1" → EXISTS (subnet-d3dcfd50...)
  ✅ subnet               "private-subnet-az2" → EXISTS (subnet-5cb4937f...)
  ✅ internet_gateway     "dev-igw" → EXISTS (igw-ac703f85...)
  ✅ nat_gateway          "dev-nat" → EXISTS (nat-5418ea87da75c060...)
  ✅ route_table          "public-route-table" → EXISTS (rtb-f4da44dc...)
  ✅ route_table          "private-route-table" → EXISTS (rtb-9425dc59...)
  ✅ security_group       "dev-web-sg" → EXISTS (sg-5c85da35302d46462...)
  ⚡ ec2_instance         "dev-web-server" → NEEDS CREATION
  1 to create, 0 to delete, 10 unchanged

  ⚡ Run 'python main.py apply' to execute this plan.
